In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path

import numpy as np

from relife.lifetime_model import SemiParametricAcceleratedFailureTime
from relife.lifetime_model._regression import LinearCovarEffect
from relife.lifetime_model._semi_parametric import SemiParamAFTData

In [ ]:
# Données chaines d'isolateur
relife_csv_datapath = Path(r"D:\Projets\RTE\ReLife\relife\relife\data\csv")
time, event, entry, *args = np.loadtxt(relife_csv_datapath / "insulator_string.csv", delimiter=",", skiprows=1,
                                       unpack=True)
covar = np.column_stack(args)

In [ ]:
# Model
model = SemiParametricAcceleratedFailureTime()

In [ ]:
# Init covar_effect
N = 100

model.covar_effect = LinearCovarEffect(
    (None,) * np.atleast_2d(np.asarray(covar, dtype=np.float64)).shape[-1]
)

# Build training_data
model._training_data = SemiParamAFTData(
    time=time[:N], covar=covar[:N], event=event[:N], entry=entry[:N]
)

In [ ]:
# Log rank
model.log_rank_stat(np.zeros(3))

In [ ]:
N = model._training_data.nb_observations
eps_time = model._log_time_residuals()
eps_entry = model._log_entry_residuals()
covar = model._training_data.covar
event = model._training_data.event

S = np.zeros(3)
for i in range(covar.shape[0]):
    X_diff_ij = covar[i, :] - covar[(i+1):, :]
    eps_time_sgn_diff_ij = np.sign(eps_time[i] - eps_time[(i+1):])
    min_eps_time = np.minimum(eps_time[i], eps_time[(i+1):])
    max_eps_entry = np.maximum(eps_entry[i], eps_entry[(i+1):])
    oij = (
            event[i] * event[(i+1):]
            + event[i] * (1 - event[(i+1):]) * (eps_time[i] < eps_time[(i+1):])
            + (1 - event[i]) * event[(i+1):] * (eps_time[i] > eps_time[(i+1):])
    )
    S -= np.sum(X_diff_ij * eps_time_sgn_diff_ij * (max_eps_entry <= min_eps_time) * oij, axis=0)

In [ ]:
np.sum(S**2) / N**2